In [1]:
import duckdb
import os
import polars as pl
import json
from IPython.display import display
import yaml

from shared_code.database.directory_creation import *
from shared_code.database.database_and_schema_creation import *
from pipelines.modrinth_api_general.schemas.modrinth_general_schemas import *

In [2]:
def read_latest_raw_project_listings(bronze_db_path:str, table_name: str) -> pl.DataFrame:
    """
    Read the latest raw project listings from the bronze database.
    """
    query = f"""
    SELECT
        run_id,
        stream,
        payload,
        c_pull_timestamp_utc
    FROM {table_name}
    ORDER BY
        c_pull_timestamp_utc DESC,
        run_id DESC
    LIMIT 1
"""

    with duckdb.connect(
        bronze_db_path,
        read_only=True,
    ) as bronze_con:
        return bronze_con.execute(query).pl()

In [16]:
def unpack_version_manifest(
    raw_df: pl.DataFrame,
) -> pl.DataFrame:

    metadata_df = raw_df.drop("payload")

    payload_rows = [
        json.loads(payload)
        for payload in raw_df["payload"].to_list()
    ]

    payload_df = pl.from_dicts(
        payload_rows,
        infer_schema_length=None,
    )

    duplicate_columns = list(
        set(metadata_df.columns)
        & set(payload_df.columns)
    )

    if duplicate_columns:
        payload_df = payload_df.drop(
            duplicate_columns
        )

    silver_df = metadata_df.hstack(
        payload_df
    )

    silver_df = (
        silver_df

        # Extract Mojang's authoritative latest IDs
        .with_columns(
            pl.col("latest")
            .struct.field("release")
            .alias("latest_release"),

            pl.col("latest")
            .struct.field("snapshot")
            .alias("latest_snapshot"),
        )

        # Convert versions list into one row per version
        .explode("versions", empty_as_null=True)

        # Convert each version struct into columns
        .unnest("versions")

        # Create derived flags
        .with_columns(
            pl.col("id")
            .eq(pl.col("latest_release"))
            .alias("is_latest_release"),

            pl.col("id")
            .eq(pl.col("latest_snapshot"))
            .alias("is_latest_snapshot"),
        )

        .drop(
            "latest",
            "latest_release",
            "latest_snapshot",
        )
    )

    return silver_df

In [4]:
def write_polars_to_silver(
    silver_db_path: str,
    df: pl.DataFrame,
    table_name: str,
) -> None:
    with duckdb.connect(silver_db_path) as silver_con:
        silver_con.register("source_df", df)

        silver_con.execute(f"""
            CREATE OR REPLACE TABLE {table_name} AS
            SELECT *
            FROM source_df
        """)

        silver_con.unregister("source_df")

In [18]:
def create_base_api_project_listings(bronze_db_path: str, silver_db_path:str, table_name: str) -> None:

    raw_df = read_latest_raw_project_listings(bronze_db_path=bronze_db_path, table_name=table_name)
    print(f"Read in {raw_df.height:,} project listings from bronze.")
    raw_count_df = (
        raw_df
        .group_by(
            "run_id",
            "stream",
        )
        .agg(
            pl.len().cast(pl.Int64).alias("row_count")
        )
    )
    total_row = pl.DataFrame({
    "run_id": [None],
    "stream": ["total_listings"],
    "row_count": [raw_count_df["row_count"].sum()],
    })

    raw_count_df = pl.concat(
        [
            raw_count_df,
            total_row,
        ],
        how="vertical",
    )

    print(raw_count_df)

    silver_df = unpack_version_manifest(raw_df)
    
    # latest_release = (
    # silver_df
    # .filter(pl.col("type") == "release")
    # .sort("releaseTime", descending=True)
    # .select("id")
    # .item(0, 0)
    # )   

    # latest_snapshot = (
    #     silver_df
    #     .filter(pl.col("type") == "snapshot")
    #     .sort("releaseTime", descending=True)
    #     .select("id")
    #     .item(0, 0)
    # )

    # silver_df = silver_df.with_columns(
    #     pl.col("id")
    #     .eq(latest_release)
    #     .alias("is_latest_release"),

    #     pl.col("id")
    #     .eq(latest_snapshot)
    #     .alias("is_latest_snapshot"),
    # )
    
    with pl.Config(
        tbl_rows = 10,
        tbl_cols = 30):

            display(silver_df)
            

    # rules_df = read_data_rules(
    #     b1=b1,
    #     table_name=b1.base_api_project_listings_table_name,
    # )

    # silver_df = apply_data_rules(
    #     df=silver_df,
    #     rules_df=rules_df,
    # )

    write_polars_to_silver(silver_db_path=silver_db_path, df=silver_df, table_name=table_name)

    print(
        f"Silver project listings written: "
        f"{silver_df.height:,}"
    )

In [19]:
config_path = "stream-config.yml"
with open(config_path, "r") as file:
    config_data = yaml.safe_load(file)

In [20]:
env = 'dev'
project_root = os.path.abspath(
    os.path.join(
        os.getcwd(),
        "..",
        "..",
        "..",
    )
)
headers = config_data['main']['headers']
ingestion_log = config_data['main']['ingestion-log']

streams = config_data['streams']
source_url = streams[0]['source-url']
table_name = streams[0]['table-name']

bronze_path = os.path.join(project_root, config_data['main']['parent-folder'], 'bronze', env, build_db_filename(streams[0]['source-url']))
silver_path = os.path.join(project_root, config_data['main']['parent-folder'], 'silver', env, build_db_filename(streams[0]['source-url']))

In [21]:
build_layer_directory(os.path.dirname(silver_path))
init_db(db_path=silver_path)
#dont need to initialize silver schemas becuase it is inferred from the polars dataframe when writing to silver

#bring clean silver data and enforce schema
create_base_api_project_listings(bronze_db_path=bronze_path, silver_db_path=silver_path, table_name=table_name)

Read in 1 project listings from bronze.
shape: (2, 3)
┌─────────────────────────────────┬─────────────────────────┬───────────┐
│ run_id                          ┆ stream                  ┆ row_count │
│ ---                             ┆ ---                     ┆ ---       │
│ str                             ┆ str                     ┆ i64       │
╞═════════════════════════════════╪═════════════════════════╪═══════════╡
│ 26baabc2-b283-442d-aed4-0a97ad… ┆ mojang_version_manifest ┆ 1         │
│ null                            ┆ total_listings          ┆ 1         │
└─────────────────────────────────┴─────────────────────────┴───────────┘


run_id,stream,c_pull_timestamp_utc,id,type,url,time,releaseTime,sha1,complianceLevel,is_latest_release,is_latest_snapshot
str,str,"datetime[μs, America/Los_Angeles]",str,str,str,str,str,str,i64,bool,bool
"""26baabc2-b283-442d-aed4-0a97ad…","""mojang_version_manifest""",2026-08-25 17:02:29.572342 PDT,"""26.3-snapshot-10""","""snapshot""","""https://piston-meta.mojang.com…","""2026-08-25T12:59:53+00:00""","""2026-08-25T12:53:43+00:00""","""46fb31ca7e74aea93545df8f1aa14b…",1,false,true
"""26baabc2-b283-442d-aed4-0a97ad…","""mojang_version_manifest""",2026-08-25 17:02:29.572342 PDT,"""26.3-snapshot-9""","""snapshot""","""https://piston-meta.mojang.com…","""2026-08-25T06:44:37+00:00""","""2026-08-17T11:46:16+00:00""","""13aee76ed25a9a334216405f8784fb…",1,false,false
"""26baabc2-b283-442d-aed4-0a97ad…","""mojang_version_manifest""",2026-08-25 17:02:29.572342 PDT,"""26.3-snapshot-8""","""snapshot""","""https://piston-meta.mojang.com…","""2026-08-25T06:44:37+00:00""","""2026-08-12T09:39:37+00:00""","""1a1a8a6c93d9bddb2882dd56204442…",1,false,false
"""26baabc2-b283-442d-aed4-0a97ad…","""mojang_version_manifest""",2026-08-25 17:02:29.572342 PDT,"""26.3-snapshot-7""","""snapshot""","""https://piston-meta.mojang.com…","""2026-08-25T06:44:37+00:00""","""2026-08-04T11:49:07+00:00""","""3b8e079dd7272f5602f9014b90b857…",1,false,false
"""26baabc2-b283-442d-aed4-0a97ad…","""mojang_version_manifest""",2026-08-25 17:02:29.572342 PDT,"""26.3-snapshot-6""","""snapshot""","""https://piston-meta.mojang.com…","""2026-08-25T06:44:37+00:00""","""2026-07-28T12:25:51+00:00""","""96508dbfbcb0f6cf1fb9889d5a8646…",1,false,false
…,…,…,…,…,…,…,…,…,…,…,…
"""26baabc2-b283-442d-aed4-0a97ad…","""mojang_version_manifest""",2026-08-25 17:02:29.572342 PDT,"""rd-161348""","""old_alpha""","""https://piston-meta.mojang.com…","""2022-03-10T09:51:38+00:00""","""2009-05-16T11:48:00+00:00""","""f22a3882d124ef4468f6eb50b12836…",0,false,false
"""26baabc2-b283-442d-aed4-0a97ad…","""mojang_version_manifest""",2026-08-25 17:02:29.572342 PDT,"""rd-160052""","""old_alpha""","""https://piston-meta.mojang.com…","""2022-03-10T09:51:38+00:00""","""2009-05-15T22:52:00+00:00""","""0cac2ceab812568826c6e5aeb4cf98…",0,false,false
"""26baabc2-b283-442d-aed4-0a97ad…","""mojang_version_manifest""",2026-08-25 17:02:29.572342 PDT,"""rd-20090515""","""old_alpha""","""https://piston-meta.mojang.com…","""2022-03-10T09:51:38+00:00""","""2009-05-14T22:00:00+00:00""","""a3165080e2b0bf20519eac5f55ee84…",0,false,false


Silver project listings written: 908
